# SobolevCellular - train (GPU, internet ON)


In [ ]:
# Train kernel (internet ON for backbone download, GPU ON). Produces weights + calib.
import os, sys, glob
comp_dir = os.path.dirname(glob.glob("/kaggle/input/*/train.csv")[0])
# code bundle (src/) mounted as a dataset; competition data as another input
bundle_root = os.path.dirname(glob.glob("/kaggle/input/*/src")[0])
sys.path.insert(0, bundle_root)
os.environ["CPM_DATA_DIR"]    = comp_dir
os.environ["CPM_WEIGHTS_DIR"] = "/kaggle/working/weights"
os.environ["CPM_FEAT_CACHE"]  = "/kaggle/working/.feat_cache"

import subprocess
BACKBONE = "imagenet_convnext_tiny"
subprocess.run([sys.executable, "-m", "src.preprocess", "--skip-masking",
                "--zip", "none", "--out", comp_dir], check=False)  # data already extracted on kaggle
subprocess.run([sys.executable, "-m", "src.train", "--backbone", BACKBONE,
                "--folds", "5", "--epochs", "150"], check=True, cwd=bundle_root)
subprocess.run([sys.executable, "-m", "src.calibrate",
                "--oof", f"{bundle_root}/oof_{BACKBONE}.npz"], check=True, cwd=bundle_root)
print("training complete -> /kaggle/working/weights")
